# 06 — Text Normalization
**Goal:** Clean messy real-world text into consistent form.

Normalization removes surface variation so that texts which differ only cosmetically compare equal: `PYTHON`, `python`, and `Python` should match; `café` and `café` should match; a bullet glyph should not leak into your tokens. Resumes arrive as PDF/DOCX exports full of curly quotes, en-dashes, ligatures, and Unicode accents — every one of those is a matching failure waiting to happen.

**Why it matters for resumes / ATS:** ATS keyword matching is string comparison. "python" vs "Python" vs "PYTHON" must collide, and a smart quote or accent in a skill name must not break the match. Normalization is the cheapest recall win in the entire pipeline — it runs before tokenization and before any matcher, and it costs one pass over the text.

## 1. Case Normalization Trade-offs

Lowercasing maximizes recall but throws away information: acronyms (`AWS`, `NLP`, `PhD`) and proper names carry meaning in their casing. The robust pattern is not blind lowercasing — it is a **canonical form map**: lowercase for lookup, canonical spelling for output.

**What the code does:** builds `term_map = {t.lower(): t for t in terms}` from a curated set, then lowercases the sample and resolves each word through the map:
- `aws` → `AWS` — canonical casing restored
- `python,` → `python,` — no match, because the trailing comma makes `python,` a different key than `python`

**Try it:** the `python,` line is the real lesson — punctuation must be stripped *before* the case lookup runs (or the map must be built on cleaned tokens). Case normalization and punctuation handling are inseparable, which is why Ch. 05's tokenization decisions feed directly into matching.

In [1]:
terms = {"Python", "NLP", "AWS", "SQL", "PhD", "B.Tech"}
term_map = {t.lower(): t for t in terms}
sample = "I know python, nlp, and aws"
for w in sample.lower().split():
    print(f"'{w}' -> canonical: '{term_map.get(w, w)}'")

'i' -> canonical: 'i'
'know' -> canonical: 'know'
'python,' -> canonical: 'python,'
'nlp,' -> canonical: 'nlp,'
'and' -> canonical: 'and'
'aws' -> canonical: 'AWS'


## 2. Unicode Normalization

The same character can be stored two ways: precomposed (`é` as one code point) or decomposed (`e` + combining accent). Add ligatures (`ﬁ`) and smart quotes, and a resume can contain several spellings of the same text. **NFKD** decomposes everything, and encoding to ASCII with `ignore` then keeps only the plain letters.

**What the code does:** for each sample, `unicodedata.normalize("NFKD", s)` followed by `.encode("ascii", "ignore").decode()`:
- `café` → `cafe` and `café` → `cafe` — the two spellings finally collide
- `ﬁle` → `file`, and smart quotes become plain quotes
- `Straße` → `Strae` — `ß` has no ASCII equivalent, so it is dropped

**Try it:** `Straße` is the lossy case — deterministic, but it will never match `Strasse`. For resume text this is usually acceptable; names and cities are the risky cases, so keep a small mapping for common non-ASCII names rather than trusting the drop.

In [2]:
import unicodedata
samples = ["caf\u00e9", "cafe\u0301", "\ufb01le", "Stra\u00dfe", "\u201csmart quotes\u201d"]
for s in samples:
    nfkd = unicodedata.normalize("NFKD", s)
    ascii_ = nfkd.encode("ascii", "ignore").decode()
    print(f"'{s}' -> '{ascii_}'")

'café' -> 'cafe'
'café' -> 'cafe'
'ﬁle' -> 'file'
'Straße' -> 'Strae'
'“smart quotes”' -> 'smart quotes'


## 3. Bullet Symbol Normalization

Resumes use `•`, `-`, `*`, and `→` interchangeably as bullet markers. If those glyphs survive into tokens, "• Python" and "- Python" never compare equal and the marker pollutes downstream matching. Normalizing all markers to one canonical symbol makes bullet lists uniform.

**What the code does:** one regex, `r'[\\s]*[•\-*→][\\s]*'`, is meant to match an optional whitespace run, any of the four marker characters, and a trailing whitespace run, replacing the match with `'> '`:
- `• Python` → `>  Python`, `- Led team` → `>  Led team`
- `* Published` → `>  Published`, `→ Reduced latency` → `>  Reduced latency`

**Read the stored output carefully:** the doubled space after the marker is a tell. The pattern in the notebook is written with doubled backslashes (`[\\s]*`), so in regex it matches literal backslash/`s` characters, not whitespace — only the bare marker matches, and the space after it survives the swap. A correct `[\s]*` would consume that space in the same pass. Same lesson as Ch. 04 and the `ResumeCleaner` below: verify regex escapes against the output you actually got.

In [3]:
import re
bullets = ["\u2022 Python", "- Led team", "* Published", "\u2192 Reduced latency"]
for b in bullets:
    print(f"'{b}' -> '{re.sub(r'[\\s]*[\u2022\-*\u2192][\\s]*', '> ', b)}'")

'• Python' -> '>  Python'
'- Led team' -> '>  Led team'
'* Published' -> '>  Published'
'→ Reduced latency' -> '>  Reduced latency'


## 4. Complete ResumeCleaner

Production normalization combines every trick above into one reusable component. A class like `ResumeCleaner` is the first thing a real resume pipeline instantiates — it runs before the tokenizer so every later stage sees consistent text.

**What the code does:** `clean()` chains four steps: NFKD-decompose, ASCII-encode with `ignore`, map bullet glyphs (`•`, `‣`, `●`) to `>`, then collapse whitespace runs with `\s+` and strip:
- input `José's résumé • Python & NLP  Sr. ML Engineer` → `Jose's resume  Python & NLP  Sr. ML Engineer`

**Read the stored output carefully:** two details are worth noticing. First, no `>` appears — the `•` was dropped by the ASCII step *before* the bullet-mapping regex runs, so that mapping is dead code in this order. Second, the doubled spaces survive — the collapse pattern in the notebook is written as `r"\\s+"` with a doubled backslash, which matches a literal backslash rather than whitespace. Fixing the escape makes the output fully clean; spotting both is exactly the output-level verification Ch. 04 warned about.

In [4]:
import unicodedata, re
class ResumeCleaner:
    def clean(self, text):
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", "ignore").decode()
        text = re.sub(r"[\u2022\u2023\u25cf]", ">", text)
        text = re.sub(r"\\s+", " ", text)
        return text.strip()

c = ResumeCleaner()
print(c.clean("Jos\u00e9's r\u00e9sum\u00e9 \u2022 Python & NLP  Sr. ML Engineer"))

Jose's resume  Python & NLP  Sr. ML Engineer


## Key Insight

**Normalization is where resume-matching recall is won or lost — and order matters as much as the steps themselves.**

Every character-level difference between a resume and a job description is a missed match: casing, Unicode spellings, bullet glyphs, whitespace. But the tools are blunt: blind lowercasing erases acronym case, ASCII-dropping eats `ß` and bullets, and a doubled backslash in a raw-string regex silently disables the step. Normalize in a fixed order — Unicode first, case via canonical maps, markers, then whitespace — and verify against real output before trusting the pipeline.

This feeds directly into Ch. 07: once the text is clean, the next question is which *words* carry meaning — and which common words should be dropped.